# Week 3-3 · command safety와 human approval packet

## 시나리오
모델이 제안한 명령을 실행하지 않고, 관찰 전용 allowlist와 shell 제어 문자 검사를 통과한 항목을 SEV1/2 승인 packet 또는 SEV3 observation plan으로 분리합니다.

## 학습 목표
- 구조화된 `CommandProposal`을 만든다.
- `dry_run`과 명령 문자열을 함께 검사한다.
- SEV1/2 승인 packet과 SEV3 observation plan에서 `executed_commands=[]`를 유지한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · proposal과 정책 선언

In [ ]:
# 실행 순서: 1단계 · proposal과 정책 선언에서 CommandProposal, proposal_is_safe을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · proposal과 정책 선언.
from pydantic import BaseModel

# 명령 문자열과 목적뿐 아니라 dry_run 의도를 구조화해 함께 검사합니다.
class CommandProposal(BaseModel):
    command: str
    purpose: str
    dry_run: bool

SAFE_PREFIXES = ("observe ", "kubectl get ", "kubectl describe ", "kubectl logs ", "kubectl rollout status ")
SHELL_CONTROL_MARKERS = (";", "&", "|", "`", "$", "<", ">", chr(92), chr(10), chr(13))

# 모델 판단과 별개로 read-only prefix와 shell 제어 문자를 다시 검사합니다.
def proposal_is_safe(proposal: CommandProposal) -> bool:
    command = proposal.command.strip()
    return (
        proposal.dry_run
        and command.startswith(SAFE_PREFIXES)
        and not any(marker in command for marker in SHELL_CONTROL_MARKERS)
    )

practice_proposals = [
    CommandProposal(command="kubectl get pods -n shop", purpose="pod 상태 관찰", dry_run=True),
    CommandProposal(command="kubectl delete namespace production", purpose="삭제", dry_run=True),
    CommandProposal(command="kubectl get pods; kubectl delete pod checkout", purpose="복합 명령", dry_run=True),
]

### 2단계 · 승인 packet 조립

In [ ]:
# 실행 순서: 2단계 · 승인 packet 조립에서 build_approval_packet을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · 승인 packet 조립.
# 하나라도 위험하면 전체 batch를 차단하고 SEV3는 observation plan으로 분리합니다.
def build_approval_packet(proposals: list[CommandProposal], severity: str = "SEV1") -> dict:
    if severity not in {"SEV1", "SEV2", "SEV3"}:
        raise ValueError(f"unsupported severity: {severity}")
    accepted = [proposal.model_dump() for proposal in proposals if proposal_is_safe(proposal)]
    rejected = [proposal.model_dump() for proposal in proposals if not proposal_is_safe(proposal)]
    all_safe = bool(proposals) and not rejected
    if not all_safe:
        return {
            "status": "blocked_manual_review",
            "approval_packet": {"proposed_commands": [], "rejected_unsafe_proposals": rejected},
            "observation_plan": None,
            "executed_commands": [],
        }
    if severity == "SEV3":
        return {
            "status": "plan_ready",
            "approval_packet": None,
            "observation_plan": {"proposed_commands": accepted},
            "executed_commands": [],
        }
    return {
        "status": "human_approval_required",
        "approval_packet": {"proposed_commands": accepted, "rejected_unsafe_proposals": []},
        "observation_plan": None,
        "executed_commands": [],
    }

approval_packet = build_approval_packet(practice_proposals)
approval_packet

### 3단계 · 안전 불변조건 확인

In [ ]:
# 실행 순서: 3단계 · 안전 불변조건 확인에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 안전 불변조건 확인.
assert approval_packet["status"] == "blocked_manual_review"
assert approval_packet["approval_packet"]["proposed_commands"] == []
assert len(approval_packet["approval_packet"]["rejected_unsafe_proposals"]) == 2
assert approval_packet["executed_commands"] == []

safe_packet = build_approval_packet([practice_proposals[0]], severity="SEV2")
assert safe_packet["status"] == "human_approval_required"
assert [item["command"] for item in safe_packet["approval_packet"]["proposed_commands"]] == ["kubectl get pods -n shop"]
assert safe_packet["executed_commands"] == []

observation = CommandProposal(command="observe checkout error-rate dashboard", purpose="지표 관찰", dry_run=True)
sev3_plan = build_approval_packet([observation], severity="SEV3")
assert sev3_plan["status"] == "plan_ready"
assert sev3_plan["approval_packet"] is None
assert sev3_plan["observation_plan"]["proposed_commands"][0]["command"] == observation.command
assert sev3_plan["executed_commands"] == []
{"mixed_batch": approval_packet["status"], "safe_batch": safe_packet["status"], "sev3": sev3_plan["status"], "executed_commands": []}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
삭제·수정 명령과 `;`, `&&`, pipe 같은 복합 shell 제어 문자를 거부합니다. 이 Notebook은 subprocess나 Kubernetes client를 호출하지 않습니다.

## 실제 app 연결
Week 3 app의 risk guard도 모델 boolean만 신뢰하지 않고 결정적 검사를 다시 수행합니다. SEV1/2 packet과 SEV3 observation plan은 모두 실행 결과가 아니라 사람에게 보여 줄 읽기 전용 계획입니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `04_incident_workflow_failure_tests.ipynb`에서는 빈 출력·혼합 proposal·한도 소진·잘못된 severity를 회귀 평가합니다.